# PySpark — Exploratory Data Analysis Notebook

This notebook demonstrates how to use PySpark for EDA inside Jupyter.

## Setup options
**Option A** – start via `pyspark` CLI (sets `spark` automatically):  
```bash
PYSPARK_DRIVER_PYTHON=jupyter PYSPARK_DRIVER_PYTHON_OPTS='notebook' pyspark --master local[*]
```

**Option B** – use `findspark` in a regular Jupyter session (run cell 1 first).

In [ ]:
# Cell 1 — only needed for Option B (skip if 'spark' is already defined)
try:
    spark  # already injected by pyspark CLI
    print('spark already available')
except NameError:
    import findspark
    findspark.init()

    from pyspark.sql import SparkSession
    spark = (SparkSession.builder
             .appName('notebook-eda')
             .master('local[*]')
             .config('spark.sql.shuffle.partitions', '4')
             .config('spark.ui.enabled', 'false')
             .getOrCreate())
    print('SparkSession created')

print('Spark version:', spark.version)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

## 1. Load data

In [ ]:
schema = StructType([
    StructField('order_id',   IntegerType(), False),
    StructField('region',     StringType(),  False),
    StructField('product',    StringType(),  False),
    StructField('quantity',   IntegerType(), False),
    StructField('unit_price', DoubleType(),  False),
])

rows = [
    (1,  'North', 'Widget-A', 10, 9.99),
    (2,  'South', 'Widget-B', 5,  19.99),
    (3,  'North', 'Widget-A', 7,  9.99),
    (4,  'East',  'Widget-C', 12, 4.99),
    (5,  'West',  'Widget-B', 3,  19.99),
    (6,  'East',  'Widget-A', 20, 9.99),
    (7,  'South', 'Widget-C', 8,  4.99),
    (8,  'North', 'Widget-B', 6,  19.99),
    (9,  'West',  'Widget-A', 15, 9.99),
    (10, 'South', 'Widget-A', 4,  9.99),
]

df = spark.createDataFrame(rows, schema)
df = df.withColumn('revenue', F.round(F.col('quantity') * F.col('unit_price'), 2))
print(f'Row count: {df.count()}')
df.printSchema()

## 2. Preview & basic stats

In [ ]:
df.show()

In [ ]:
df.describe('quantity', 'unit_price', 'revenue').show()

## 3. Revenue by region

In [ ]:
region_summary = (df
    .groupBy('region')
    .agg(
        F.count('order_id').alias('num_orders'),
        F.sum('quantity').alias('total_units'),
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
    )
    .orderBy(F.desc('total_revenue')))

region_summary.show()

## 4. Top products by revenue (SQL)

In [ ]:
df.createOrReplaceTempView('orders')

spark.sql("""
    SELECT product,
           SUM(quantity)          AS total_units,
           ROUND(SUM(revenue), 2) AS total_revenue
    FROM   orders
    GROUP  BY product
    ORDER  BY total_revenue DESC
""").show()

## 5. Window function — rank products within each region

In [ ]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy('region').orderBy(F.desc('revenue'))

(df
 .withColumn('rank', F.rank().over(window_spec))
 .filter(F.col('rank') == 1)
 .select('region', 'product', 'revenue', 'rank')
 .show())

## 6. Write & read Parquet

In [ ]:
region_summary.write.mode('overwrite').parquet('/tmp/notebook_output')
spark.read.parquet('/tmp/notebook_output').show()

In [ ]:
# Always stop the session when done
spark.stop()
print('SparkSession stopped')